In [16]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans
import networkx as nx
import matplotlib.pyplot as plt

In [17]:

# previously checked if there are any non-symmetric, non-binary, or non-zero-diagonal entries
def load_data(file_path: Path) -> sparse.csr_matrix:
    A = np.loadtxt(file_path, delimiter=",", dtype=np.int8)
    A = sparse.csr_matrix(A)
    return A


def spectral_clustering(A, K, random_state=42, return_embedding=False):
    n = A.shape[0]
    deg = np.asarray(A.sum(axis=1)).ravel().astype(float)
    inv_sqrt_deg = np.zeros_like(deg)
    nz = deg > 0
    inv_sqrt_deg[nz] = 1.0 / np.sqrt(deg[nz])

    D_inv_sqrt = sparse.diags(inv_sqrt_deg, offsets=0, format="csr")
    S = D_inv_sqrt @ A @ D_inv_sqrt
    L = sparse.identity(n, format="csr") - S

    _, vecs = eigsh(L, k=K, which="SM")

    row_norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1.0
    Y = vecs / row_norms

    km = KMeans(n_clusters=K, n_init=20, random_state=random_state)
    labels = km.fit_predict(Y) + 1

    if return_embedding:
        return labels, Y
    return labels


def save_labels_csv(out_path: Path, labels_1based: np.ndarray) -> None:
    with out_path.open("w", encoding="utf-8") as f:
        for i, c in enumerate(labels_1based, start=1):
            f.write(f"{i},{int(c)}\n")


def plot_graph_clusters(A_csr, labels, out_path, seed=42, title=None):
    n = A_csr.shape[0]
    G = nx.Graph()
    G.add_nodes_from(range(n))

    rows, cols = A_csr.nonzero()
    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    pos = nx.spring_layout(G, seed=seed)

    plt.figure(figsize=(8, 8))
    nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.8)
    nx.draw_networkx_nodes(
        G, pos,
        node_size=60,
        node_color=labels,
    )

    plt.axis("off")
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def plot_embedding(Y, labels, out_path, title=None):
    if Y.shape[1] < 2:
        return

    plt.figure(figsize=(7, 6))
    plt.scatter(Y[:, 0], Y[:, 1], c=labels, s=40)
    if title:
        plt.title(title)
    plt.xlabel("Embedding dim 1")
    plt.ylabel("Embedding dim 2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


In [18]:

GROUP_NAME = "KnownK_results"
AUTHORS = "Jakub Oganowski, Hubert Jaczyński"
REPO_URL = "https://github.com/OganKuba/P10-Social"

INPUT_DIR = Path(r"competition")
OUTPUT_DIR = Path(GROUP_NAME)
FIGS_DIR = OUTPUT_DIR / "figs"
RANDOM_STATE = 42

def save_labels_csv(out_path: Path, labels_1based: np.ndarray) -> None:
    with out_path.open("w", encoding="utf-8") as f:
        for i, c in enumerate(labels_1based, start=1):
            f.write(f"{i},{int(c)}\n")

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGS_DIR.mkdir(parents=True, exist_ok=True)

    jobs = [
        ("D1-K=2.csv", 2),
        ("D2-K=7.csv", 7),
        ("D3-K=12.csv", 12),
    ]

    timings = []

    for fname, K in jobs:
        A = load_data(INPUT_DIR / fname)

        t0 = time.perf_counter()
        labels, Y = spectral_clustering(A, K, random_state=RANDOM_STATE, return_embedding=True)
        t1 = time.perf_counter()
        elapsed = t1 - t0

        save_labels_csv(OUTPUT_DIR / fname, labels)

        stem = Path(fname).stem
        plot_graph_clusters(
            A, labels,
            FIGS_DIR / f"{stem}_graph.png",
            seed=RANDOM_STATE,
            title=f"{stem} (K={K})"
        )

        plot_embedding(
            Y, labels,
            FIGS_DIR / f"{stem}_embedding.png",
            title=f"{stem} spectral embedding"
        )

        timings.append((fname, elapsed))
        print(f"{fname}: time={elapsed:.6f}s")

    with (OUTPUT_DIR / "description.txt").open("w", encoding="utf-8") as f:
        f.write(f"{AUTHORS}\n")
        f.write(f"{REPO_URL}\n")
        for fname, elapsed in timings:
            f.write(f"{{{fname}, {elapsed:.6f}s}}\n")


if __name__ == "__main__":
    main()


D1-K=2.csv: time=0.039069s
D2-K=7.csv: time=0.044095s
D3-K=12.csv: time=0.049675s
